# Análise da Geração de Energia Solar

## 1. Inspeção inicial

In [1]:
import pandas as pd

In [2]:
# Carregamento dos dados
df_plant_1_generation = pd.read_csv('../datasets/Plant_1_Generation_Data.csv')
df_plant_1_weather = pd.read_csv('../datasets/Plant_1_Weather_Sensor_Data.csv')
df_plant_2_generation = pd.read_csv('../datasets/Plant_2_Generation_Data.csv')
df_plant_2_weather = pd.read_csv('../datasets/Plant_2_Weather_Sensor_Data.csv')

### 1. Dados de geração da usina 1

In [3]:
df_plant_1_generation.shape

(68778, 7)

In [4]:
df_plant_1_generation.columns

Index(['DATE_TIME', 'PLANT_ID', 'SOURCE_KEY', 'DC_POWER', 'AC_POWER',
       'DAILY_YIELD', 'TOTAL_YIELD'],
      dtype='str')

In [5]:
df_plant_1_generation.dtypes

DATE_TIME          str
PLANT_ID         int64
SOURCE_KEY         str
DC_POWER       float64
AC_POWER       float64
DAILY_YIELD    float64
TOTAL_YIELD    float64
dtype: object

In [6]:
df_plant_1_generation

,DATE_TIME,PLANT_ID,SOURCE_KEY,DC_POWER,AC_POWER,DAILY_YIELD,TOTAL_YIELD
0,15-05-2020 00:00,4135001,1BY6WEcLGh8j5v7,0.0,0.0,0.000,6259559.0
1,15-05-2020 00:00,4135001,1IF53ai7Xc0U56Y,0.0,0.0,0.000,6183645.0
2,15-05-2020 00:00,4135001,3PZuoBAID5Wc2HD,0.0,0.0,0.000,6987759.0
3,15-05-2020 00:00,4135001,7JYdWkrLSPkdwr4,0.0,0.0,0.000,7602960.0
4,15-05-2020 00:00,4135001,McdE0feGgRqW7Ca,0.0,0.0,0.000,7158964.0
...,...,...,...,...,...,...,...
68773,17-06-2020 23:45,4135001,uHbuxQJl8lW7ozc,0.0,0.0,5967.000,7287002.0
68774,17-06-2020 23:45,4135001,wCURE6d3bPkepu2,0.0,0.0,5147.625,7028601.0
68775,17-06-2020 23:45,4135001,z9Y9gH1T5YWrNuG,0.0,0.0,5819.000,7251204.0
68776,17-06-2020 23:45,4135001,zBIq5rxdHJRwDNY,0.0,0.0,5817.000,6583369.0


In [7]:
df_plant_1_generation.index

RangeIndex(start=0, stop=68778, step=1)

In [8]:
df_plant_1_generation["PLANT_ID"].nunique()

1

In [9]:
df_plant_1_generation["SOURCE_KEY"].nunique()

22

In [10]:
df_plant_1_generation["DATE_TIME"].nunique()

3158

In [11]:
df_plant_1_generation.duplicated().sum()

np.int64(0)

In [12]:
df_plant_1_generation.duplicated(
    subset=["PLANT_ID", "SOURCE_KEY", "DATE_TIME"]
).sum()

np.int64(0)

In [13]:
df_plant_1_generation["DATE_TIME"].dtype

<StringDtype(storage='python', na_value=nan)>

#### Observações iniciais

- O conjunto possui 68778 linhas e 7 colunas.

- Cada linha parece representar o registro feito por um microinversor específico da primeira usina em um intervalo de tempo de 15 minutos.

- O arquivo contém um único valor distinto em PLANT_ID, evidenciando que todos os registros realmente são da mesma usina.

- Foram identificados 22 inversores distintos em SOURCE_KEY.

- A coluna DATE_TIME foi interpretada como string.

- A combinação das colunas PLANT_ID, SOURCE_KEY e DATE_TIME não apresenta duplicatas, portanto é uma candidata à identificação única de cada medição.

- DC_POWER parece representar a potência em corrente contínua produzida pelos painéis.

- AC_POWER parece representar a potência em corrente alternada após a conversão pelo inversor.

- A coluna DAILY_YIELD parece representar a energia acumulada ao longo do dia até o momento da medição, já TOTAL_YIELD parece representar a energia total acumulada pelo inversor.

#### Investigação da frequência temporal dos registros

Considerando medições em intervalos de 15 minutos, seriam esperados 3264 horários distintos durante esse período. Porém, foram encontrados apenas 3158 horários, indicando a ausência de 106 horários de medição. Além disso, se todos os 22 inversores tivessem registros em todos esses horários, o esperado seria 69476 registros, mas o conjunto contém apenas 68778. Sendo assim, faz-se necessário investigar esses pontos com mais profundidade.

In [14]:
# Criação de uma cópia do dataset com a coluna date time convertida para o formato correto

df_plant_1_generation_analysis = (
    df_plant_1_generation.copy()
)

df_plant_1_generation_analysis['DATE_TIME'] = (
    pd.to_datetime(
        df_plant_1_generation_analysis['DATE_TIME'], format="%d-%m-%Y %H:%M"
    )
)

In [15]:
df_plant_1_generation_analysis['DATE_TIME'].min()

Timestamp('2020-05-15 00:00:00')

In [16]:
df_plant_1_generation_analysis['DATE_TIME'].max()

Timestamp('2020-06-17 23:45:00')

In [17]:
df_plant_1_generation["SOURCE_KEY"].value_counts()

SOURCE_KEY
bvBOhCH3iADSZry    3155
1BY6WEcLGh8j5v7    3154
7JYdWkrLSPkdwr4    3133
VHMLBKoKgIrUVDU    3133
ZnxXDlPa8U1GXgE    3130
ih0vzX44oOqAx2f    3130
wCURE6d3bPkepu2    3126
z9Y9gH1T5YWrNuG    3126
iCRJl6heRkivqQ3    3125
pkci93gMrogZuBj    3125
uHbuxQJl8lW7ozc    3125
McdE0feGgRqW7Ca    3124
rGa61gmuvPhdLxV    3124
sjndEbLyjtCKgGv    3124
zVJPv84UY57bAof    3124
ZoEaEvLYb1n2sOq    3123
1IF53ai7Xc0U56Y    3119
adLQvlD726eNBSB    3119
zBIq5rxdHJRwDNY    3119
3PZuoBAID5Wc2HD    3118
WRmjgnKYAwPKWDb    3118
YxYtjZvoooNbGkE    3104
Name: count, dtype: int64

In [18]:
unique_date_times = df_plant_1_generation_analysis['DATE_TIME'].drop_duplicates().sort_values()

In [19]:
time_intervals = unique_date_times.diff()

In [20]:
time_intervals.value_counts()

DATE_TIME
0 days 00:15:00    3148
0 days 00:30:00       2
0 days 03:00:00       1
0 days 01:00:00       1
0 days 04:15:00       1
0 days 09:00:00       1
0 days 01:45:00       1
0 days 08:00:00       1
0 days 00:45:00       1
Name: count, dtype: int64

In [21]:
time_gap_analysis = pd.DataFrame({
    'previous_date_time': unique_date_times.shift(1),
    'current_date_time': unique_date_times,
    'interval': time_intervals
})

time_gap_analysis[
    time_gap_analysis['interval'] > pd.Timedelta(minutes=15)
]

,previous_date_time,current_date_time,interval
1954,2020-05-15 23:00:00,2020-05-16 02:00:00,0 days 03:00:00
9146,2020-05-19 11:30:00,2020-05-19 12:30:00,0 days 01:00:00
11290,2020-05-20 13:15:00,2020-05-20 17:30:00,0 days 04:15:00
11774,2020-05-20 22:45:00,2020-05-21 07:45:00,0 days 09:00:00
15632,2020-05-23 05:00:00,2020-05-23 06:45:00,0 days 01:45:00
16952,2020-05-23 21:30:00,2020-05-23 22:00:00,0 days 00:30:00
19728,2020-05-25 05:30:00,2020-05-25 06:00:00,0 days 00:30:00
27404,2020-05-28 22:15:00,2020-05-29 06:15:00,0 days 08:00:00
67260,2020-06-17 06:00:00,2020-06-17 06:45:00,0 days 00:45:00


In [22]:
inverter_periods = (
    df_plant_1_generation_analysis
    .groupby('SOURCE_KEY')['DATE_TIME']
    .agg(['min', 'max', 'count'])
)

inverter_periods

,min,max,count
SOURCE_KEY,,,
1BY6WEcLGh8j5v7,2020-05-15 00:00:00,2020-06-17 23:45:00,3154
1IF53ai7Xc0U56Y,2020-05-15 00:00:00,2020-06-17 23:45:00,3119
3PZuoBAID5Wc2HD,2020-05-15 00:00:00,2020-06-17 23:45:00,3118
7JYdWkrLSPkdwr4,2020-05-15 00:00:00,2020-06-17 23:45:00,3133
McdE0feGgRqW7Ca,2020-05-15 00:00:00,2020-06-17 23:45:00,3124
VHMLBKoKgIrUVDU,2020-05-15 00:00:00,2020-06-17 23:45:00,3133
WRmjgnKYAwPKWDb,2020-05-15 00:00:00,2020-06-17 23:45:00,3118
YxYtjZvoooNbGkE,2020-05-15 01:00:00,2020-06-17 23:45:00,3104
ZnxXDlPa8U1GXgE,2020-05-15 00:00:00,2020-06-17 23:45:00,3130


#### Observações da investigação de frequência temporal

- O conjunto de dados abrange o período entre 15/05/2020 às 00:00 e 17/06/2020 às 23:45, totalizando 34 dias registrados.

- A maior parte dos horários consecutivos apresenta o intervalo esperado de 15 minutos. Entretanto, foram identificadas nove lacunas temporais maiores, variando entre 30 minutos e 9 horas. Desse modo, fica evidente porque existem exatamente os 106 horários ausentes no período analisado.

- As maiores interrupções ocorreram entre 20/05/2020 às 22:45 e 21/05/2020 às 07:45, com duração de 9 horas, e entre 28/05/2020 às 22:15 e 29/05/2020 às 06:15, com duração de 8 horas.

- A quantidade de registros por inversor também varia entre os 22 inversores. O inversor com mais medições possui 3.155 registros, enquanto o inversor com menos medições possui 3.104 registros.

- Existem 698 combinações ausentes entre inversor e horário.

- Com exceção do inversor 'YxYtjZvoooNbGkE', que apresenta o primeiro registro em 15/05/2020 às 01:00, todos os inversores apresentam registros desde 15/05/2020 às 00:00. Todos possuem como último registro o horário de 17/06/2020 às 23:45. Sendo assim, as diferenças na quantidade de medições não parece ser originada por períodos de funcionamento distintos entre inversores, mas sim estar relacionados principalmente a ausências distribuidas ao longo do período observado.

#### Verificação de qualidade dos valores

In [23]:
missing_values = df_plant_1_generation.isna().sum()

missing_values[
    missing_values > 0
]

Series([], dtype: int64)

In [24]:
numeric_columns = [
    'DC_POWER',
    'AC_POWER',
    'DAILY_YIELD',
    'TOTAL_YIELD'
]

In [25]:
(df_plant_1_generation[numeric_columns] < 0).sum()

DC_POWER       0
AC_POWER       0
DAILY_YIELD    0
TOTAL_YIELD    0
dtype: int64

In [26]:
(df_plant_1_generation[numeric_columns] == 0).sum()

DC_POWER       31951
AC_POWER       31951
DAILY_YIELD    18696
TOTAL_YIELD        0
dtype: int64

In [27]:
# Verificando a porcentagem do conjunto que é igual a zero

(df_plant_1_generation[numeric_columns] == 0).sum() / df_plant_1_generation.shape[0] * 100

DC_POWER       46.455262
AC_POWER       46.455262
DAILY_YIELD    27.183111
TOTAL_YIELD     0.000000
dtype: float64

In [28]:
df_plant_1_generation[numeric_columns].describe()

,DC_POWER,AC_POWER,DAILY_YIELD,TOTAL_YIELD
count,68778.000000,68778.000000,68778.000000,6.877800e+04
mean,3147.426211,307.802752,3295.968737,6.978712e+06
std,4036.457169,394.396439,3145.178309,4.162720e+05
min,0.000000,0.000000,0.000000,6.183645e+06
25%,0.000000,0.000000,0.000000,6.512003e+06
50%,429.000000,41.493750,2658.714286,7.146685e+06
75%,6366.964286,623.618750,6274.000000,7.268706e+06
max,14471.125000,1410.950000,9163.000000,7.846821e+06


#### Resultado inicial da qualidade dos valores

- O conjunto não possui valores ausentes explícitos nem valores negativos.

- As colunas DC_POWER e AC_POWER possuem 31.951 registros iguais a zero, correspondentes a aproximadamente 46.46% do conjunto. Provavelmente, parte desses valores está associada aos períodos sem geração solar.

- A média de DC_POWER é aproximadamente 3147.43, enquanto sua mediana é 429.00. Em AC_POWER, a média é aproximadamente 307.80 e a mediana é 41.49. A diferença entre média e mediana, juntamente com a grande quantidade de zeros, indica que existe uma grande concentração de valores baixos e presença de medições consideravelmente mais elevadas que puxam a média para cima.

- Os valores de DC_POWER e AC_POWER apresentam escalas bem distintas, o que pode indicar perda na conversão, mas ainda é necessário analisar mais a fundo.

- A coluna DAILY_YIELD possui 18.696 registros iguais a zero, correspondentes a aproximadamente 27.18% do conjunto. Provavelmente esses registros estão relacionados ao início do ciclo de geração diário.

- A colunas TOTAL_YIELD não possui valores iguais a zero. Seus valores variam entre aproximadamente 6.18 milhões e 7.85 milhões.

### 2. Dados climáticos da usina 1

In [29]:
df_plant_1_weather.shape

(3182, 6)

In [30]:
df_plant_1_weather.columns

Index(['DATE_TIME', 'PLANT_ID', 'SOURCE_KEY', 'AMBIENT_TEMPERATURE',
       'MODULE_TEMPERATURE', 'IRRADIATION'],
      dtype='str')

In [31]:
df_plant_1_weather.dtypes

DATE_TIME                  str
PLANT_ID                 int64
SOURCE_KEY                 str
AMBIENT_TEMPERATURE    float64
MODULE_TEMPERATURE     float64
IRRADIATION            float64
dtype: object

In [32]:
df_plant_1_weather

,DATE_TIME,PLANT_ID,SOURCE_KEY,AMBIENT_TEMPERATURE,MODULE_TEMPERATURE,IRRADIATION
0,2020-05-15 00:00:00,4135001,HmiyD2TTLFNqkNe,25.184316,22.857507,0.0
1,2020-05-15 00:15:00,4135001,HmiyD2TTLFNqkNe,25.084589,22.761668,0.0
2,2020-05-15 00:30:00,4135001,HmiyD2TTLFNqkNe,24.935753,22.592306,0.0
3,2020-05-15 00:45:00,4135001,HmiyD2TTLFNqkNe,24.846130,22.360852,0.0
4,2020-05-15 01:00:00,4135001,HmiyD2TTLFNqkNe,24.621525,22.165423,0.0
...,...,...,...,...,...,...
3177,2020-06-17 22:45:00,4135001,HmiyD2TTLFNqkNe,22.150570,21.480377,0.0
3178,2020-06-17 23:00:00,4135001,HmiyD2TTLFNqkNe,22.129816,21.389024,0.0
3179,2020-06-17 23:15:00,4135001,HmiyD2TTLFNqkNe,22.008275,20.709211,0.0
3180,2020-06-17 23:30:00,4135001,HmiyD2TTLFNqkNe,21.969495,20.734963,0.0


In [33]:
df_plant_1_weather.index

RangeIndex(start=0, stop=3182, step=1)

In [34]:
df_plant_1_weather['PLANT_ID'].nunique()

1

In [37]:
df_plant_1_weather['SOURCE_KEY'].nunique()

1

In [39]:
df_plant_1_weather['DATE_TIME'].nunique()

3182

#### Verificação de chave candidata

In [40]:
df_plant_1_weather.duplicated().sum()

np.int64(0)

In [59]:
df_plant_1_weather.duplicated(
    subset=["PLANT_ID", "DATE_TIME"]
).sum()

np.int64(0)

In [58]:
df_plant_1_weather.duplicated(
    subset=["PLANT_ID", "SOURCE_KEY", "DATE_TIME"]
).sum()

np.int64(0)

#### Observações iniciais
- O conjunto possui 3182 linhas e 6 colunas.
- Cada linha parece representar uma medição meteorológica da primeira usina em determinado instante.
- O arquivo contém apeanas um valor distinto em `PLANT_ID`, o que confirma que todos os registros pertence à mesma usina.
- Foi identificado apenas um sensor distintos em `SOURCE_KEY`.
- A coluna `DATE_TIME` foi interpretada como string e precisará ser convertida para um tipo temporal antes das análises. Além disso, `PLANT_ID` foi interpretada como int, o que pode tornar necessário realizar uma conversão para string já que se trata de um identificador. 
- A combinação `PLANT_ID`, `SOURCE_KEY` e `DATE_TIME` não apresenta duplicatas.
- A combinação `PLANT_ID` e `DATE_TIME` não apresenta duplicatas.
- `AMBIENT_TEMPERATURE` representa a temperatura ambiente registrada pelo sensor.
- `MODULE_TEMPERATURE` representa a temperatura dos módulos solares.
- `IRRADIATION` representa a irradiação solar registrada no momento da medição.

Como existe apenas um sensor na primeira usina, a combinação PLANT_ID + DATE_TIME já identifica cada medição meteorológica. Entretanto, a inclusão de SOURCE_KEY pode tornar a chave mais adequada à identificação do equipamento e permitir a representação de mais de um sensor por usina se necessário.

#### Conversão temporal

In [42]:
df_plant_1_weather_analysis = df_plant_1_weather.copy()

In [43]:
df_plant_1_weather_analysis['DATE_TIME'] = pd.to_datetime(
    df_plant_1_weather_analysis['DATE_TIME'],
    format="%Y-%m-%d %H:%M:%S"
)

In [44]:
df_plant_1_weather_analysis['DATE_TIME'].min()

Timestamp('2020-05-15 00:00:00')

In [45]:
df_plant_1_weather_analysis['DATE_TIME'].max()

Timestamp('2020-06-17 23:45:00')

In [46]:
weather_date_times = (
    df_plant_1_weather_analysis['DATE_TIME']
    .drop_duplicates()
    .sort_values()
)

In [47]:
weather_time_intervals = weather_date_times.diff()

In [48]:
weather_time_intervals.value_counts()

DATE_TIME
0 days 00:15:00    3173
0 days 00:30:00       2
0 days 03:00:00       1
0 days 01:00:00       1
0 days 04:15:00       1
0 days 07:15:00       1
0 days 01:30:00       1
0 days 04:30:00       1
Name: count, dtype: int64

In [49]:
weather_gap_analysis = pd.DataFrame({
    "previous_date_time": weather_date_times.shift(1),
    "current_date_time": weather_date_times,
    "interval": weather_time_intervals,
    })

In [50]:
weather_gap_analysis[
    weather_gap_analysis["interval"] > pd.Timedelta(minutes=15)
]   

,previous_date_time,current_date_time,interval
93,2020-05-15 23:00:00,2020-05-16 02:00:00,0 days 03:00:00
420,2020-05-19 11:30:00,2020-05-19 12:30:00,0 days 01:00:00
520,2020-05-20 13:15:00,2020-05-20 17:30:00,0 days 04:15:00
549,2020-05-21 00:30:00,2020-05-21 07:45:00,0 days 07:15:00
732,2020-05-23 05:15:00,2020-05-23 06:45:00,0 days 01:30:00
792,2020-05-23 21:30:00,2020-05-23 22:00:00,0 days 00:30:00
1288,2020-05-29 01:45:00,2020-05-29 06:15:00,0 days 04:30:00
1799,2020-06-03 13:45:00,2020-06-03 14:15:00,0 days 00:30:00


#### Qualidade dos valores

In [51]:
missing_values_weather_1 = df_plant_1_weather.isna().sum()

missing_values_weather_1[
    missing_values_weather_1 > 0
]

Series([], dtype: int64)

In [52]:
weather_numeric_columns = [
    "AMBIENT_TEMPERATURE",
    "MODULE_TEMPERATURE",
    "IRRADIATION"
]

df_plant_1_weather[weather_numeric_columns].describe()

,AMBIENT_TEMPERATURE,MODULE_TEMPERATURE,IRRADIATION
count,3182.000000,3182.000000,3182.000000
mean,25.531606,31.091015,0.228313
std,3.354856,12.261222,0.300836
min,20.398505,18.140415,0.000000
25%,22.705182,21.090553,0.000000
50%,24.613814,24.618060,0.024653
75%,27.920532,41.307840,0.449588
max,35.252486,65.545714,1.221652


In [53]:
(df_plant_1_weather[weather_numeric_columns] < 0).sum()

AMBIENT_TEMPERATURE    0
MODULE_TEMPERATURE     0
IRRADIATION            0
dtype: int64

In [54]:
(df_plant_1_weather[weather_numeric_columns] == 0).sum()

AMBIENT_TEMPERATURE       0
MODULE_TEMPERATURE        0
IRRADIATION            1425
dtype: int64

#### Comparação temporal com a geração

In [ ]:
# Verificando a quantidade de horários em comum entre o arquivo de geração e o arquivo de registros climáticos

generation_date_times = set(
    df_plant_1_generation_analysis['DATE_TIME']
)

weather_date_times_set = set(
    df_plant_1_weather_analysis['DATE_TIME']
)

common_date_times = (
    generation_date_times & weather_date_times_set
)

len(common_date_times)

3157

In [56]:
# Verificando os horários de geração sem registro meteorológicos

generation_without_weather = (generation_date_times - weather_date_times_set)

len(generation_without_weather)

1

In [57]:
# Verificando os horários meteorológicos sem registro de geração

weather_without_generation = (weather_date_times_set - generation_date_times)

len(weather_without_generation)

25

### 3. Dados de geração da usina 2

In [61]:
df_plant_2_generation.shape


(67698, 7)

In [62]:
df_plant_2_generation.dtypes

DATE_TIME          str
PLANT_ID         int64
SOURCE_KEY         str
DC_POWER       float64
AC_POWER       float64
DAILY_YIELD    float64
TOTAL_YIELD    float64
dtype: object

In [63]:
df_plant_2_generation

,DATE_TIME,PLANT_ID,SOURCE_KEY,DC_POWER,AC_POWER,DAILY_YIELD,TOTAL_YIELD
0,2020-05-15 00:00:00,4136001,4UPUqMRk7TRMgml,0.0,0.0,9425.000000,2.429011e+06
1,2020-05-15 00:00:00,4136001,81aHJ1q11NBPMrL,0.0,0.0,0.000000,1.215279e+09
2,2020-05-15 00:00:00,4136001,9kRcWv60rDACzjR,0.0,0.0,3075.333333,2.247720e+09
3,2020-05-15 00:00:00,4136001,Et9kgGMDl729KT4,0.0,0.0,269.933333,1.704250e+06
4,2020-05-15 00:00:00,4136001,IQ2d7wF4YD8zU1Q,0.0,0.0,3177.000000,1.994153e+07
...,...,...,...,...,...,...,...
67693,2020-06-17 23:45:00,4136001,q49J1IKaHRwDQnt,0.0,0.0,4157.000000,5.207580e+05
67694,2020-06-17 23:45:00,4136001,rrq4fwE8jgrTyWY,0.0,0.0,3931.000000,1.211314e+08
67695,2020-06-17 23:45:00,4136001,vOuJvMaM2sgwLmb,0.0,0.0,4322.000000,2.427691e+06
67696,2020-06-17 23:45:00,4136001,xMbIugepa2P7lBB,0.0,0.0,4218.000000,1.068964e+08


#### Duplicatas e chave candidata

In [64]:
df_plant_2_generation["PLANT_ID"].nunique()

1

In [65]:
df_plant_2_generation["SOURCE_KEY"].nunique()

22

In [66]:
df_plant_2_generation["DATE_TIME"].nunique()

3259

In [67]:
df_plant_2_generation.duplicated().sum()

np.int64(0)

In [68]:
df_plant_2_generation.duplicated(
    subset=["PLANT_ID", "SOURCE_KEY", "DATE_TIME"]
).sum()

np.int64(0)

#### Análise temporal

In [69]:
df_plant_2_generation_analysis = (
    df_plant_2_generation.copy()
    )

df_plant_2_generation_analysis["DATE_TIME"] = pd.to_datetime(
    df_plant_2_generation_analysis["DATE_TIME"],
    format="%Y-%m-%d %H:%M:%S",
)

In [70]:
df_plant_2_generation_analysis["DATE_TIME"].min()

Timestamp('2020-05-15 00:00:00')

In [71]:
df_plant_2_generation_analysis["DATE_TIME"].max()

Timestamp('2020-06-17 23:45:00')

In [73]:
plant_2_generation_date_times = (
    df_plant_2_generation_analysis["DATE_TIME"]
    .drop_duplicates()
    .sort_values()
)

plant_2_generation_intervals = (
    plant_2_generation_date_times.diff()
)

plant_2_generation_intervals.value_counts()

DATE_TIME
0 days 00:15:00    3253
0 days 00:30:00       5
Name: count, dtype: int64

#### Qualidade dos dados

In [75]:
missing_values_plant_2_generation = (
    df_plant_2_generation.isna().sum()
)

missing_values_plant_2_generation[
    missing_values_plant_2_generation > 0
]

Series([], dtype: int64)

In [76]:
generation_numeric_columns = [
    "DC_POWER",
    "AC_POWER",
    "DAILY_YIELD",
    "TOTAL_YIELD",
]

df_plant_2_generation[
    generation_numeric_columns
].describe()

,DC_POWER,AC_POWER,DAILY_YIELD,TOTAL_YIELD
count,67698.000000,67698.000000,67698.000000,6.769800e+04
mean,246.701961,241.277825,3294.890295,6.589448e+08
std,370.569597,362.112118,2919.448386,7.296678e+08
min,0.000000,0.000000,0.000000,0.000000e+00
25%,0.000000,0.000000,272.750000,1.996494e+07
50%,0.000000,0.000000,2911.000000,2.826276e+08
75%,446.591667,438.215000,5534.000000,1.348495e+09
max,1420.933333,1385.420000,9873.000000,2.247916e+09


In [77]:
(df_plant_2_generation[generation_numeric_columns] < 0).sum()

DC_POWER       0
AC_POWER       0
DAILY_YIELD    0
TOTAL_YIELD    0
dtype: int64

In [78]:
(df_plant_2_generation[generation_numeric_columns] == 0).sum()

DC_POWER       35662
AC_POWER       35662
DAILY_YIELD    11569
TOTAL_YIELD      563
dtype: int64

#### Comparação entre DC e AC

In [86]:
df_plant_2_generation["AC_DC_RATIO"] = (
    df_plant_2_generation["AC_POWER"] / df_plant_2_generation["DC_POWER"]
)

In [87]:
df_plant_2_generation["AC_DC_RATIO"].describe()

count    32036.000000
mean         0.976806
std          0.005020
min          0.912790
25%          0.975014
50%          0.978432
75%          0.980247
max          1.008320
Name: AC_DC_RATIO, dtype: float64

#### Resultados
- O conjunto possui 67698 linhas e 7 colunas.
- Foi identificado um único valor em `PLANT_ID` e 22 inversores em `SOURCE_KEY`.
- O período registrado vai de 2020-05-15 00:00:00 até 2020-06-17 23:45:00
- O intervalo temporal predominante é de 15 minutos.
- A combinação `PLANT_ID + SOURCE_KEY + DATE_TIME` não apresenta duplicatas.
- Não foram encontrados valores ausentes nem valores negativos.
- Os valores máximos de `DC_POWER` e `AC_POWER` são, respectivamente, 1420.933333 e 1385.420000
- Diferentemente da usina 1, `DC_POWER` e `AC_POWER` apresentam escalas próximas na usina 2.
- A razão entre `AC_POWER` e `DC_POWER`, apresenta mediana de aproximadamente 0.98
- A diferença entre as escalas das duas usinas reforça a hipótese de inconsistência de escala em `DC_POWER` na usina 1.

### 4. Dados climáticos da usina 2

In [88]:
df_plant_2_weather.shape

(3259, 6)

In [89]:
df_plant_2_weather.dtypes

DATE_TIME                  str
PLANT_ID                 int64
SOURCE_KEY                 str
AMBIENT_TEMPERATURE    float64
MODULE_TEMPERATURE     float64
IRRADIATION            float64
dtype: object

In [90]:
df_plant_2_weather

,DATE_TIME,PLANT_ID,SOURCE_KEY,AMBIENT_TEMPERATURE,MODULE_TEMPERATURE,IRRADIATION
0,2020-05-15 00:00:00,4136001,iq8k7ZNt4Mwm3w0,27.004764,25.060789,0.0
1,2020-05-15 00:15:00,4136001,iq8k7ZNt4Mwm3w0,26.880811,24.421869,0.0
2,2020-05-15 00:30:00,4136001,iq8k7ZNt4Mwm3w0,26.682055,24.427290,0.0
3,2020-05-15 00:45:00,4136001,iq8k7ZNt4Mwm3w0,26.500589,24.420678,0.0
4,2020-05-15 01:00:00,4136001,iq8k7ZNt4Mwm3w0,26.596148,25.088210,0.0
...,...,...,...,...,...,...
3254,2020-06-17 22:45:00,4136001,iq8k7ZNt4Mwm3w0,23.511703,22.856201,0.0
3255,2020-06-17 23:00:00,4136001,iq8k7ZNt4Mwm3w0,23.482282,22.744190,0.0
3256,2020-06-17 23:15:00,4136001,iq8k7ZNt4Mwm3w0,23.354743,22.492245,0.0
3257,2020-06-17 23:30:00,4136001,iq8k7ZNt4Mwm3w0,23.291048,22.373909,0.0


In [91]:
df_plant_2_weather["PLANT_ID"].nunique()

1

In [92]:
df_plant_2_weather["SOURCE_KEY"].nunique()

1

In [93]:
df_plant_2_weather["DATE_TIME"].nunique()

3259

In [94]:
df_plant_2_weather.duplicated().sum()

np.int64(0)

In [95]:
df_plant_2_weather.duplicated(
    subset=["PLANT_ID", "SOURCE_KEY", "DATE_TIME"]
).sum()

np.int64(0)

In [96]:
df_plant_2_weather.duplicated(
    subset=["PLANT_ID", "DATE_TIME"]
).sum()

np.int64(0)

#### Análise temporal

In [97]:
df_plant_2_weather_analysis = (
    df_plant_2_weather.copy()
)

df_plant_2_weather_analysis["DATE_TIME"] = pd.to_datetime(
    df_plant_2_weather_analysis["DATE_TIME"],
    format="%Y-%m-%d %H:%M:%S",
)

In [98]:
df_plant_2_weather_analysis["DATE_TIME"].min()

Timestamp('2020-05-15 00:00:00')

In [99]:
df_plant_2_weather_analysis["DATE_TIME"].max()

Timestamp('2020-06-17 23:45:00')

In [100]:
plant_2_weather_date_times = (
    df_plant_2_weather_analysis["DATE_TIME"]
    .drop_duplicates()
    .sort_values()
)

plant_2_weather_intervals = (
    plant_2_weather_date_times.diff()
)

plant_2_weather_intervals.value_counts()

DATE_TIME
0 days 00:15:00    3253
0 days 00:30:00       5
Name: count, dtype: int64

In [101]:
plant_2_weather_gap_analysis = pd.DataFrame({
    "previous_date_time": plant_2_weather_date_times.shift(1),
    "current_date_time": plant_2_weather_date_times,
    "interval": plant_2_weather_intervals,
})

plant_2_weather_gap_analysis[
    plant_2_weather_gap_analysis["interval"] > pd.Timedelta(minutes=15)
]

,previous_date_time,current_date_time,interval
93,2020-05-15 23:00:00,2020-05-15 23:30:00,0 days 00:30:00
447,2020-05-19 15:45:00,2020-05-19 16:15:00,0 days 00:30:00
1406,2020-05-29 15:45:00,2020-05-29 16:15:00,0 days 00:30:00
1700,2020-06-01 17:30:00,2020-06-01 18:00:00,0 days 00:30:00
1876,2020-06-03 13:45:00,2020-06-03 14:15:00,0 days 00:30:00


#### Qualidade dos valores

In [102]:
missing_values_plant_2_weather = (
    df_plant_2_weather.isna().sum()
)

missing_values_plant_2_weather[
    missing_values_plant_2_weather > 0
]

Series([], dtype: int64)

In [103]:
weather_numeric_columns = [
    "AMBIENT_TEMPERATURE",
    "MODULE_TEMPERATURE",
    "IRRADIATION",
]

df_plant_2_weather[
    weather_numeric_columns
].describe()

,AMBIENT_TEMPERATURE,MODULE_TEMPERATURE,IRRADIATION
count,3259.000000,3259.000000,3259.000000
mean,28.069400,32.772408,0.232737
std,4.061556,11.344034,0.312693
min,20.942385,20.265123,0.000000
25%,24.602135,23.716881,0.000000
50%,26.981263,27.534606,0.019040
75%,31.056757,40.480653,0.438717
max,39.181638,66.635953,1.098766


In [104]:
(df_plant_2_weather[weather_numeric_columns] < 0).sum()

AMBIENT_TEMPERATURE    0
MODULE_TEMPERATURE     0
IRRADIATION            0
dtype: int64

In [105]:
(df_plant_2_weather[weather_numeric_columns] == 0).sum()

AMBIENT_TEMPERATURE       0
MODULE_TEMPERATURE        0
IRRADIATION            1397
dtype: int64

#### Comparação temporal com a geração da usina 2

In [106]:
plant_2_generation_date_times_set = set(
    df_plant_2_generation_analysis["DATE_TIME"]
)

plant_2_weather_date_times_set = set(
    df_plant_2_weather_analysis["DATE_TIME"]
)

In [107]:
plant_2_common_date_times = (
    plant_2_generation_date_times_set & plant_2_weather_date_times_set
)

len(plant_2_common_date_times)

3259

In [108]:
plant_2_generation_without_weather = (
    plant_2_generation_date_times_set - plant_2_weather_date_times_set
)

len(plant_2_generation_without_weather)

0

In [109]:
plant_2_weather_without_generation = (
    plant_2_weather_date_times_set - plant_2_generation_date_times_set
)

len(plant_2_weather_without_generation)

0

#### Resultados
- O conjunto possui 3259 linhas e 6 colunas.
- Foi identificado um único valor em `PLANT_ID` e um único sensor em `SOURCE_KEY`.
- O período registrado vai de 2020-05-15 00:00:00 até 2020-06-17 23:45:00.
- O intervalo temporal predominante é de 15 minutos.
- Foram encontradas 5 lacunas superiores a 15 minutos.
- A combinação `PLANT_ID + SOURCE_KEY + DATE_TIME` não apresenta duplicatas.
- A combinação `PLANT_ID + DATE_TIME` não apresenta duplicatas.
- Não foram encontrados valores ausentes e valores negativos.
- Foram identificados 3259 horários compartilhados com os dados de geração.
- Não existem horários de geração sem medição meteorológica.
- Não existem horários meteorológicos sem medição de geração.

### Resumo da inspeção inicial
- Os quatro arquivos possuem registros de duas usinas solares.
- Os arquivos de geração possuem medições por inversor e horário.
- Os arquivos meteorológicos possuem medições por sensor e horário.
- Os arquivos equivalentes podem ser concatenados, preservando `PLANT_ID`.
- `DATE_TIME` deverá ser convertido para um tipo temporal.
- `SOURCE_KEY` deverá ser preservado como identificador original.
- Serão criados códigos amigáveis adicionais para inversores e sensores.
- A combinação `PLANT_ID + SOURCE_KEY + DATE_TIME` é candidata à chave primária composta.
- Existem lacunas temporais que deverão ser preservadas e documentadas.
- A escala de `DC_POWER` da usina 1 precisa ser investigada e tratada de forma documentada.
- Os horários de geração e clima não devem ser considerados perfeitamente correspondentes antes da validação.

## 2. Plano de preparação dos dados

Com base na inspeção inicial dos quatro arquivos CSV, foram definidas as transformação necessárias antes da inserção dos dados no MySQL. 

Os arquivos originais serão preservados sem alterações, portanto as alterações serão realizadas em cópias dos DataFrames e serão reproduzidas depois pelo processo responsável pela injeção no banco.

### 2.1 Conversão da coluna `DATE_TIME`

- A coluna `DATE_TIME` de todos os arquivos será convertida de string para um time de data e hora.
- Os valores serão armazenados no banco de dados como DATETIME.
- Como se trata de um dataset referente a duas usinas localizadas na Índia, não será necessário levar em consideração fuso horário.

### 2.2 Padronização dos nomes das colunas

Os nomes das colunas serão padronizados em snake_case antes da injeção no banco.

#### Dados de geração
- `DATE_TIME` → `date_time`
- `PLANT_ID` → `plant_id`
- `SOURCE_KEY` → `source_key`
- `DC_POWER` → `dc_power`
- `AC_POWER` → `ac_power`
- `DAILY_YIELD` → `daily_yield`
- `TOTAL_YIELD` → `total_yield`

#### Dados meteorológicos
- `DATE_TIME` → `date_time`
- `PLANT_ID` → `plant_id`
- `SOURCE_KEY` → `source_key`
- `AMBIENT_TEMPERATURE` → `ambient_temperature`
- `MODULE_TEMPERATURE` → `module_temperature`
- `IRRADIATION` → `irradiation`

### 2.3 Concatenação dos arquivos equivalentes

#### Tabela de geração
Serão concatenados:
- `Plant_1_Generation_Data.csv`
- `Plant_2_Generation_Data.csv`
 
#### Tabela meteorológica
Serão concatenados:
- `Plant_1_Weather_Sensor_Data.csv`
- `Plant_2_Weather_Sensor_Data.csv`

A coluna `plant_id` será preservada para identificar a usina de origem de cada registro.

O índice do Pandas não será utilizado como identificador no banco.

### 2.4 Códigos amigáveis para equipamentos

Nos dados de geração, SOURCE_KEY identifica um inversor, mas, nos dados meteorológicos, SOURCE_KEY identifica um sensor climático. Portanto, serão criados códigos adicionais para facilitar consultas, interpretações e apresentações, mas os identificadores originais serão mantidos para garantir a rastreabilidade entre o banco e os arquivos CSV.

#### Inversores
Exemplos:
- `INV-P01-01`
- `INV-P01-02`
- `INV-P02-01`
#### Sensores
Exemplos:
- `SEN-P01-01`
- `SEN-P02-01`
 
Os códigos serão adicionados em novas colunas:
- `inverter_code` na tabela de geração.
- `sensor_code` na tabela meteorológica.
 
A numeração seguirá uma regra reproduzível: os valores únicos de `SOURCE_KEY` serão ordenados antes da geração dos códigos, garantindo que o mesmo equipamento sempre receba o mesmo código durante uma nova execução.

### 2.5 Preservação das lacunas temporais e valores iguais a zero

Foram identificadas lacunas temporais e combinações ausentes entre equipamentos e horários, as quais serão preservadas durante a preparação e a injeção no banco.
 

As lacunas serão documentadas como uma limitação e poderão ser analisadas durante a etapa de análise estatística.


Valores iguais a zero também serão mantidos porque não significam automaticamente um valor ausente, por exemplo em colunas referentes à geração ou irradiação.

### Investigação da escala de `DC_POWER`

Foi identificada uma diferença de escala entre `DC_POWER` e `AC_POWER` nos dados da primeira usina. `DC_POWER` apresenta valores aproximadamente dez vezes maiores que `AC_POWER`. 

Observando os dados da segunda usina, as duas medidas aparecem em escalas próximas e, como as duas variáveis são descritas na mesma unidade, essa diferença pode indicar uma inconsistência de escala nos dados de `DC_POWER` da primeira usina.

Até que essa hipótese seja validada os valores originais serão preservados. Caso seja necessário ajustar os valores, a coluna original será preservada e uma coluna adicional corrigida será criada.